# 深度学习 HW04
## 序列模型、循环神经网络、嵌入向量和注意力机制

**学号：** 20234080415  
**姓名：** 唐宇正  
**Date:** 2026-06-24

---
# 2 序列模型

## 2.1 理论计算题

给定字符序列 `ababc`，采用一阶马尔可夫模型 $p(x_t\mid x_{t-1})$，词汇表为 $\{a,b,c\}$，使用拉普拉斯平滑（加 1 平滑）。

序列中的相邻转移为：

$$a\to b,\quad b\to a,\quad a\to b,\quad b\to c$$

从字符 `b` 出发的转移共有 2 次：

$$b\to a:1,\quad b\to c:1,\quad b\to b:0$$

词汇表大小 $|V|=3$。拉普拉斯平滑公式为：

$$p(x\mid b)=\frac{\text{count}(b\to x)+1}{\sum_{v\in V}\text{count}(b\to v)+|V|}$$

因此分母为：

$$2+3=5$$

### 1. $p(a\mid b)$

$$p(a\mid b)=\frac{1+1}{5}=\boxed{\frac25=0.4}$$

### 2. $p(c\mid b)$

$$p(c\mid b)=\frac{1+1}{5}=\boxed{\frac25=0.4}$$

## 2.2 编程题：文本预处理与 n-gram 样本生成

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """文本清洗、分词、构建词表，并生成长度为 n 的特征序列与下一个词标签。"""
    cleaned = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    tokens = cleaned.split()

    counter = Counter(tokens)
    vocab_words = sorted(counter.keys(), key=lambda w: (-counter[w], w))
    vocab = {word: idx for idx, word in enumerate(vocab_words)}

    features, labels = [], []
    for i in range(len(tokens) - n + 1):
        features.append(tokens[i:i + n])
        next_pos = i + n
        labels.append(tokens[next_pos] if next_pos < len(tokens) else None)

    return vocab, features, labels

vocab, features, labels = preprocess_text('The time machine', n=2)
print('词汇表:', vocab)
print('特征:', features)
print('标签:', labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time'], ['time', 'machine']]
标签: ['machine', None]


---
# 3 循环神经网络

## 3.1 理论计算题

线性 RNN 定义为：

$$h_t=W_{hh}h_{t-1}+W_{hx}x_t$$

$$o_t=W_{oh}h_t$$

平方损失为：

$$L=\frac12\sum_{t=1}^{T}(o_t-y_t)^2$$

令输出层误差为：

$$e_t=\frac{\partial L}{\partial o_t}=o_t-y_t$$

由于 $h_t$ 不仅影响当前输出 $o_t$，还影响未来的隐藏状态，所以需要通过时间反向传播。定义：

$$\delta_t=\frac{\partial L}{\partial h_t}$$

则有递推式：

$$\delta_t=W_{oh}^{\top}e_t+W_{hh}^{\top}\delta_{t+1},\quad \delta_{T+1}=0$$

对 $W_{hh}$ 的梯度为每个时间步贡献之和：

$$\boxed{\frac{\partial L}{\partial W_{hh}}=\sum_{t=1}^{T}\delta_t h_{t-1}^{\top}}$$

展开递推可以看出：

$$\delta_t=\sum_{k=t}^{T}(W_{hh}^{\top})^{k-t}W_{oh}^{\top}e_k$$

因此梯度中包含 $W_{hh}$ 的连乘项。如果 $W_{hh}$ 的谱半径或最大奇异值长期小于 1，连乘项会趋近于 0，出现梯度消失；如果长期大于 1，连乘项会快速增大，出现梯度爆炸。

## 3.2 编程题：简单 RNN 单元的前向传播与单步反向传播

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a_t)
    cache = (x_t, h_prev, W_hx, W_hh, a_t)
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, a_t = cache
    da = dh_next * (1 - np.tanh(a_t) ** 2)
    dx_t = da @ W_hx.T
    dh_prev = da @ W_hh.T
    dW_hx = x_t.T @ da
    dW_hh = h_prev.T @ da
    db_h = da.sum(axis=0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

np.random.seed(42)
batch_size, input_size, hidden_size = 2, 3, 4
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.zeros(hidden_size)
dh_next = np.random.randn(batch_size, hidden_size)

h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

print('h_t shape:', h_t.shape)
print('dx_t shape:', dx_t.shape)
print('dh_prev shape:', dh_prev.shape)
print('dW_hx shape:', dW_hx.shape)
print('dW_hh shape:', dW_hh.shape)
print('db_h shape:', db_h.shape)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (4,)


---
# 4 高级循环神经网络

## 4.1 理论计算题

假设深度双向 RNN 有 $L$ 层，每层每个方向的隐藏单元数为 $H$，输入维度为 $D$，最终输出维度为 $O$，只考虑最后输出层。

单向普通 RNN 每层参数包括：输入到隐藏层权重、隐藏到隐藏层权重和隐藏层偏置。

### 第 1 层

每个方向的输入维度为 $D$，参数量为：

$$DH+H^2+H$$

双向后为：

$$2(DH+H^2+H)$$

### 第 2 到第 $L$ 层

从第二层开始，每个时间步的输入是上一层双向隐藏状态拼接，维度为 $2H$。每个方向参数量为：

$$2H\cdot H+H^2+H=3H^2+H$$

共 $L-1$ 层、双向，因此参数量为：

$$2(L-1)(3H^2+H)$$

### 最后输出全连接层

最后序列表示维度为 $2H$，输出维度为 $O$，参数量为：

$$2HO+O$$

所以总参数量为：

$$\boxed{2(DH+H^2+H)+2(L-1)(3H^2+H)+2HO+O}$$

## 4.2 编程题：双向 RNN 编码器

In [3]:
import torch
from torch import nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
        )

    def forward(self, X):
        outputs, h_n = self.rnn(X)
        # h_n: (num_layers * 2, batch, hidden_dim)，最后两项分别是最后一层的正向和反向状态
        forward_last = h_n[-2]
        backward_last = h_n[-1]
        final_state = torch.cat([forward_last, backward_last], dim=-1)
        return outputs, final_state

seq_len, batch, input_dim, hidden_dim = 5, 3, 4, 6
encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=2)
X = torch.randn(seq_len, batch, input_dim)
outputs, final_state = encoder(X)

print('每个时间步的拼接隐藏状态:', tuple(outputs.shape))
print('最终序列表示:', tuple(final_state.shape))

每个时间步的拼接隐藏状态: (5, 3, 12)
最终序列表示: (3, 12)


---
# 5 嵌入向量

## 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$。设中心词向量为 $v_c$，正样本上下文词向量为 $u_o$，第 $k$ 个负样本词向量为 $u_{n_k}$。使用负采样时，目标是提高真实上下文词出现的概率，同时降低负样本词作为上下文出现的概率。

对数似然目标为：

$$\log \sigma(u_o^{\top}v_c)+\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{\top}v_c)$$

因此最小化的损失函数为：

$$\boxed{L=-\log \sigma(u_o^{\top}v_c)-\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{\top}v_c)}$$

完整目标函数可以写为：

$$\boxed{J=\sum_{(c,o)\in D}\left[-\log \sigma(u_o^{\top}v_c)-\sum_{k=1}^{K}\log \sigma(-u_{n_k}^{\top}v_c)\right]}$$

负样本通常从噪声分布 $P_n(w)$ 中采样。常见选择是基于词频的分布，例如：

$$P_n(w)=\frac{f(w)^{3/4}}{\sum_{w'\in V}f(w')^{3/4}}$$

这样既保留高频词更容易被采样的特点，又不会让极高频词占据过大比例。

## 5.2 编程题：CBOW 完整 softmax 前向传播与损失

In [4]:
import torch.nn.functional as F

def cbow_forward_loss(context_indices, target_indices, W, W_out):
    """CBOW前向传播：上下文词向量取平均，再用完整softmax预测中心词。"""
    context_embeds = W[context_indices]             # (batch, context_size, d)
    hidden = context_embeds.mean(dim=1)             # (batch, d)
    logits = hidden @ W_out                         # (batch, V)
    loss = F.cross_entropy(logits, target_indices)
    probs = F.softmax(logits, dim=1)
    return loss, probs, hidden

torch.manual_seed(0)
V, d, context_size, batch_size = 8, 5, 4, 3
W = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)
context_indices = torch.tensor([[0, 1, 2, 3],
                                [2, 3, 4, 5],
                                [1, 4, 6, 7]])
target_indices = torch.tensor([4, 1, 3])

loss, probs, hidden = cbow_forward_loss(context_indices, target_indices, W, W_out)
loss.backward()

print('hidden shape:', tuple(hidden.shape))
print('probs shape:', tuple(probs.shape))
print('loss:', float(loss))
print('W.grad shape:', tuple(W.grad.shape))
print('W_out.grad shape:', tuple(W_out.grad.shape))

hidden shape: (3, 5)
probs shape: (3, 8)
loss: 2.8625290393829346
W.grad shape: (8, 5)
W_out.grad shape: (5, 8)


---
# 6 注意力机制

## 6.1 理论计算题

给定查询矩阵：

$$Q\in \mathbb{R}^{2\times4}$$

键矩阵：

$$K\in \mathbb{R}^{3\times4}$$

值矩阵：

$$V\in \mathbb{R}^{3\times5}$$

缩放点积注意力使用：

$$score=\frac{QK^{\top}}{\sqrt{d_k}},\quad d_k=4$$

因此：

$$S=\frac{QK^{\top}}{2}\in \mathbb{R}^{2\times3}$$

对每一行做 softmax 得到注意力权重矩阵：

$$A=\text{softmax}(S)\in \mathbb{R}^{2\times3}$$

最后对值矩阵加权求和：

$$\boxed{O=AV\in \mathbb{R}^{2\times5}}$$

也就是说，第 $i$ 个查询的输出为：

$$o_i=\sum_{j=1}^{3}\frac{\exp(q_i^{\top}k_j/2)}{\sum_{m=1}^{3}\exp(q_i^{\top}k_m/2)}v_j$$

题目只给出了矩阵维度而没有给出具体元素，因此不能得到唯一的数值矩阵；上式就是完整的符号计算过程。

## 6.2 编程题：实现 Multi-Head Attention 前向传播

In [5]:
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch_size, _ = X.shape

        def project(layer, x):
            y = layer(x)  # (seq_len, batch, d_model)
            y = y.permute(1, 0, 2)  # (batch, seq_len, d_model)
            y = y.reshape(batch_size, seq_len, self.num_heads, self.d_k)
            return y.permute(0, 2, 1, 3)  # (batch, heads, seq_len, d_k)

        Q = project(self.W_q, X)
        K = project(self.W_k, X)
        V = project(self.W_v, X)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)  # (batch, heads, seq_len, d_k)

        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.reshape(batch_size, seq_len, self.d_model)
        context = context.permute(1, 0, 2)  # (seq_len, batch, d_model)
        output = self.W_o(context)
        return output, attn

mha = MultiHeadAttention(d_model=4, num_heads=2)
X = torch.randn(6, 3, 4)
Y, attn = mha(X)
print('输入形状:', tuple(X.shape))
print('输出形状:', tuple(Y.shape))
print('注意力权重形状:', tuple(attn.shape))

输入形状: (6, 3, 4)
输出形状: (6, 3, 4)
注意力权重形状: (3, 2, 6, 6)


---
# 总结

本次作业完成了马尔可夫模型平滑估计、文本 n-gram 样本生成、RNN 反向传播推导与单步实现、双向 RNN 编码器、Skip-gram 负采样目标、CBOW 完整 softmax 损失，以及缩放点积注意力和多头注意力的实现。